# Khối B — Train YOLOv8s trên VAIPE (6 lớp, imgsz 1024)

**Cài đặt notebook (cột phải):** Accelerator = **GPU T4 x2**, Internet = **On**, Add Input = dataset `vaipe-yolo`.

1. Chạy thử: để `MODE = "smoke"` → **Run All** (tương tác) → xem ô cuối báo `SMOKE OK` + ước lượng thời gian.
2. Chạy thật: đổi `MODE = "full"` → **Save Version → Save & Run All (Commit)** để chạy nền.
3. Tải về: tab Output của version → `v8s_1024_out.zip` (chứa `best.pt`, `results.csv`, hình).

In [ ]:
# ==== CẤU HÌNH ====
MODE = "smoke"          # "smoke" = chạy thử tập con | "full" = train thật
IMGSZ = 1024
MODEL = "yolov8s.pt"
EPOCHS_FULL = 50
BATCH_FULL = 16         # OOM -> 8 (KHÔNG giảm IMGSZ)
DEVICE = "0,1"          # T4 x2; nếu chỉ có 1 GPU -> "0"
SEED = 42
SMOKE_N_TRAIN, SMOKE_N_VAL, SMOKE_EPOCHS = 64, 32, 2

In [ ]:
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPUs", torch.cuda.device_count())
assert torch.cuda.is_available(), "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2"

In [ ]:
# Tìm dataset (Kaggle có thể lồng thêm thư mục) và copy sang /kaggle/working (input chỉ đọc -> lỗi ghi .cache)
import glob, os, shutil, random, json, time, subprocess, sys
from pathlib import Path

ymls = sorted(glob.glob("/kaggle/input/**/vaipe.yaml", recursive=True))
assert ymls, "Không thấy vaipe.yaml dưới /kaggle/input — đã Add Input dataset chưa?"
SRC = Path(ymls[0]).parent
print("Dataset gốc:", SRC)

WORK = Path("/kaggle/working/vaipe")
if not WORK.exists():
    shutil.copytree(SRC, WORK)
for s, n in (("train", 820), ("valid", 176)):
    ni = len(list((WORK/"images"/s).glob("*.png"))); nl = len(list((WORK/"labels"/s).glob("*.txt")))
    print(f"{s}: {ni} ảnh, {nl} nhãn"); assert ni == nl == n, f"{s} sai số lượng"

NAMES = ["drugname", "quantity", "usage", "diagnose", "date", "other"]
def write_yaml(root, path):
    txt = f"path: {root}\ntrain: images/train\nval: images/valid\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES))
    Path(path).write_text(txt); return str(path)
DATA_FULL = write_yaml(WORK, "/kaggle/working/vaipe.yaml")
print(open(DATA_FULL).read())

In [ ]:
# Hàm chạy lệnh yolo, in log trực tiếp (dùng CLI để DDP 2 GPU chạy ổn trong notebook)
def run(cmd):
    print(">>", cmd, flush=True); t = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end="")
    p.wait(); dt = time.time() - t
    assert p.returncode == 0, f"Lệnh lỗi (mã {p.returncode})"
    return dt

COMMON = f"seed={SEED} fliplr=0 mosaic=0.5 patience=15 device={DEVICE} project=/kaggle/working/runs exist_ok=True plots=True"

In [ ]:
# ==== CHẠY THỬ: tập con nhỏ, 2 epoch ====
if MODE == "smoke":
    rng = random.Random(SEED)
    SM = Path("/kaggle/working/vaipe_smoke")
    shutil.rmtree(SM, ignore_errors=True)
    for s, n in (("train", SMOKE_N_TRAIN), ("valid", SMOKE_N_VAL)):
        (SM/"images"/s).mkdir(parents=True); (SM/"labels"/s).mkdir(parents=True)
        for img in rng.sample(sorted((WORK/"images"/s).glob("*.png")), n):
            shutil.copy(img, SM/"images"/s/img.name)
            shutil.copy(WORK/"labels"/s/(img.stem + ".txt"), SM/"labels"/s/(img.stem + ".txt"))
    DATA_SMOKE = write_yaml(SM, "/kaggle/working/vaipe_smoke.yaml")

    dt = run(f"yolo detect train model={MODEL} data={DATA_SMOKE} imgsz={IMGSZ} epochs={SMOKE_EPOCHS} batch=8 {COMMON} name=smoke")

    run_dir = Path("/kaggle/working/runs/smoke")
    for f in ("weights/best.pt", "weights/last.pt", "results.csv"):
        assert (run_dir/f).exists(), f"Thiếu {f}"
    print(open(run_dir/"results.csv").read())

    # Thử nạp weights + suy luận 1 ảnh valid -> kiểm tra luồng infer cho detect/yolo.py
    from ultralytics import YOLO
    m = YOLO(str(run_dir/"weights/best.pt"))
    img = sorted((SM/"images"/"valid").glob("*.png"))[0]
    r = m.predict(str(img), imgsz=IMGSZ, conf=0.01, verbose=False)[0]
    print("names:", m.names); print("số box (conf 0.01):", len(r.boxes))
    assert list(m.names.values()) == NAMES, "Thứ tự lớp sai!"

    # Ước lượng thời gian train thật (thô: gồm cả overhead khởi động, nên là cận trên)
    per_img_epoch = dt / (SMOKE_EPOCHS * SMOKE_N_TRAIN)
    est_h = per_img_epoch * 820 * EPOCHS_FULL / 3600
    print(f"\nSMOKE OK — {dt:.0f}s cho {SMOKE_EPOCHS} epoch x {SMOKE_N_TRAIN} ảnh")
    print(f"Ước lượng thô train thật ({EPOCHS_FULL} epoch x 820 ảnh): ~{est_h:.1f} h (cận trên)")
    print("-> Đổi MODE = 'full' rồi Save Version -> Save & Run All (Commit).")

In [ ]:
# ==== TRAIN THẬT ====
if MODE == "full":
    dt = run(f"yolo detect train model={MODEL} data={DATA_FULL} imgsz={IMGSZ} epochs={EPOCHS_FULL} batch={BATCH_FULL} save_period=10 {COMMON} name=v8s_{IMGSZ}")
    print(f"Train xong sau {dt/3600:.2f} h")
    # Nếu session bị ngắt giữa chừng: chạy lại với lệnh
    #   yolo detect train resume model=/kaggle/working/runs/v8s_1024/weights/last.pt

In [ ]:
# ==== ĐÁNH GIÁ VALID + ĐÓNG GÓI ====
if MODE == "full":
    from ultralytics import YOLO
    run_dir = Path(f"/kaggle/working/runs/v8s_{IMGSZ}")
    m = YOLO(str(run_dir/"weights/best.pt"))
    met = m.val(data=DATA_FULL, split="val", imgsz=IMGSZ, device=0, plots=True, project=str(run_dir), name="val_best", exist_ok=True)
    summary = {"mAP50": float(met.box.map50), "mAP50_95": float(met.box.map),
               "per_class_mAP50": {NAMES[i]: float(v) for i, v in enumerate(met.box.ap50)},
               "ultralytics": ultralytics.__version__, "imgsz": IMGSZ, "epochs": EPOCHS_FULL,
               "batch": BATCH_FULL, "model": MODEL, "seed": SEED}
    (run_dir/"config.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
    print(json.dumps(summary, indent=2))
    shutil.make_archive(f"/kaggle/working/v8s_{IMGSZ}_out", "zip", run_dir)
    shutil.rmtree("/kaggle/working/vaipe", ignore_errors=True)   # bớt Output, chỉ giữ kết quả
    print("Tải: Output -> v8s_%d_out.zip" % IMGSZ)